In [8]:
KEYWORDS = [
    "technology", "science", "education", "coding",
    "news", "finance", "gaming", "music", "movies", "sports"
]


In [10]:
from googleapiclient.discovery import build
import os
from dotenv import load_dotenv
load_dotenv()


True

In [11]:
API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build('youtube', 'v3', developerKey=API_KEY)
print("YouTube API client created successfully.")

YouTube API client created successfully.


In [12]:
def search_channels_by_keyword(youtube, keyword, max_channels=50):
    channel_ids = []
    next_page_token = None

    while len(channel_ids) < max_channels:
        request = youtube.search().list(
            part="id,snippet",
            q=keyword,
            type="channel",
            maxResults=50,
            pageToken=next_page_token
        )

        response = request.execute()

        for item in response["items"]:
            channel_ids.append(item["id"]["channelId"])
            if len(channel_ids) >= max_channels:
                break

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return channel_ids


In [13]:

all_channel_ids = []

for keyword in KEYWORDS:
    print(f"Searching channels for keyword: {keyword}")
    ids = search_channels_by_keyword(youtube, keyword, max_channels=50)
    all_channel_ids.extend(ids)


Searching channels for keyword: technology
Searching channels for keyword: science
Searching channels for keyword: education
Searching channels for keyword: coding
Searching channels for keyword: news
Searching channels for keyword: finance
Searching channels for keyword: gaming
Searching channels for keyword: music
Searching channels for keyword: movies
Searching channels for keyword: sports


In [15]:
print(len(all_channel_ids), "channel IDs extracted.")

500 channel IDs extracted.


In [16]:
all_channel_ids = list(set(all_channel_ids))
len(all_channel_ids)


499

In [17]:
def classify_channel(youtube, channel_id):
    response = youtube.channels().list(
        part="statistics",
        id=channel_id
    ).execute()

    if not response["items"]:
        return None

    subs = int(response["items"][0]["statistics"].get("subscriberCount", 0))

    if subs >= 1_000_000:
        return "big"
    elif subs >= 100_000:
        return "medium"
    else:
        return "small"


In [18]:
channel_records = []

for cid in all_channel_ids:
    category = classify_channel(youtube, cid)
    if category:
        channel_records.append({
            "channel_id": cid,
            "category": category
        })


In [19]:
import pandas as pd

df = pd.DataFrame(channel_records)
df.to_csv("../data/channel_ids.csv", index=False)
